# S14 · From prices to returns, and what real markets look like

We turn the wandering price from notebook 1 into **returns** (the percent change,
day to day), see why quants use log-returns, and then reproduce two famous facts
about real markets with our own eyes: fat tails and volatility clustering.

**New here? Read this once.**

- New to Python? Run each cell top to bottom and read the note above it. You do not
  need to write any code yourself.
- New to the ideas? A return is just "how much the price changed today, in percent".
  The one-page background is `primers/returns_and_log_returns.md`, and the histogram
  and bell-curve picture is `primers/what_is_a_distribution.md`.
- Already know returns and stylised facts? Look for the **Stretch (optional)** cells.
- Any unfamiliar word is in `primers/glossary.md`.

## Setup

If you are on **Google Colab**, run the next cell once. On your **own machine**
everything is already installed, so it does nothing there.

In [ ]:
# This notebook uses numpy, pandas and matplotlib (Colab has them) plus
# yfinance for the live download, which Colab lacks. Install only that, only
# on Colab.
import sys
if "google.colab" in sys.modules:
    !pip install -q yfinance
else:
    print("Not on Colab - assuming the libraries are already installed.")

In [ ]:
import numpy as np                  # fast maths on lists of numbers
import pandas as pd                   # tables of data, indexed by date
import matplotlib.pyplot as plt       # drawing charts

# Same seed as notebook 1 so the offline fallback is identical.
np.random.seed(0)

## Step 1 — get the same price series as notebook 1

We reuse the exact same recipe: try to download the Nifty 50 with `yfinance`, and
fall back to a seeded synthetic series if we're offline. This is the same block you
already read in notebook 1, so we won't re-explain every line.

In [ ]:
# Try live Nifty 50 data; fall back to a seeded synthetic price series offline.
ticker_symbol = "^NSEI"
close_price = None
data_source = ""

try:
    import yfinance as yf
    downloaded = yf.download(
        ticker_symbol,
        start="2021-01-01",
        end="2025-01-01",
        auto_adjust=True,
        progress=False,
    )
    if downloaded is None or len(downloaded) == 0:
        raise ValueError("no data returned")
    close_price = downloaded["Close"].dropna().squeeze()
    data_source = "LIVE download from yfinance (" + ticker_symbol + ")"

except Exception:
    # OFFLINE FALLBACK: build a realistic geometric random walk.
    print("Could not download live data; using a synthetic price series instead.")
    number_of_days = 1000
    daily_drift = 0.0003
    daily_volatility = 0.012
    daily_shocks = np.random.normal(daily_drift, daily_volatility, size=number_of_days)
    starting_price = 15000.0
    price_levels = starting_price * np.exp(np.cumsum(daily_shocks))
    dates = pd.bdate_range(start="2021-01-01", periods=number_of_days)
    close_price = pd.Series(price_levels, index=dates, name="Close")
    data_source = "SYNTHETIC fallback (seeded geometric random walk)"

print("Data source:", data_source)
print("Number of days:", len(close_price))

## Step 2 — why returns, not prices?

A **return** is the percent change in price from one day to the next. We study
returns instead of raw prices for three reasons:

- Prices have no natural scale. A ₹100 stock and a ₹3,000 stock aren't comparable,
  but a 2% move means the same thing for either.
- Returns are comparable across stocks and across time.
- Returns are far steadier and better-behaved (we'll see this in a moment).

The **simple return** is just `(today − yesterday) / yesterday`. pandas computes it
in one call with `.pct_change()`.

In [ ]:
# .pct_change() computes (today - yesterday) / yesterday for each day.
# The very first day has no "yesterday", so it is NaN; we drop it.
simple_returns = close_price.pct_change().dropna()

print("First 5 simple returns (as %):")
print((simple_returns.head() * 100).round(3))

## Step 3 — log-returns, the quant's choice

The **log-return** is the natural log of the price ratio:

`log_return = ln(price_today / price_yesterday)`

It is the same day-to-day move as the simple return, just written with a logarithm.
Quants prefer it for two reasons:

1. **It adds up over time.** A week's log-return is simply the *sum* of its daily
   log-returns. (Simple returns have to be multiplied, which is messier.)
2. **It is symmetric.** A `+x` and a `−x` move are true mirror images.

If the logarithm is unfamiliar, don't worry: it's just a button that turns the
"multiply" version of returns into an "add" version. The picture in
`primers/returns_and_log_returns.md` is enough.

In [ ]:
# np.log is the natural logarithm. close_price.shift(1) moves yesterday's
# value onto today's row, so the ratio is today / yesterday.
price_ratio = close_price / close_price.shift(1)
log_returns = np.log(price_ratio).dropna()

print("First 5 log-returns (as %):")
print((log_returns.head() * 100).round(3))

## Step 4 — simple vs log returns nearly agree

For ordinary daily moves (a percent or two), the two definitions give almost the
same number. We line them up on the same dates and look at the biggest gap between
them; it should be tiny.

In [ ]:
# The absolute gap between the two definitions on each day.
difference = (simple_returns - log_returns).abs()

print("Largest difference between simple and log returns:",
      round(float(difference.max()), 5))
print("In words: for everyday daily moves they are practically the same.")

## Step 5 — plot the returns

Now plot the returns over time and compare the shape to the price from notebook 1.
Unlike the wandering price, the return series **hovers around zero** with a roughly
steady spread. This stability is exactly why we model returns.

In [ ]:
plt.figure(figsize=(9, 4.5))
plt.plot(log_returns.index, log_returns.values * 100, color="#C0392B", linewidth=0.8)
plt.axhline(0, color="grey", linewidth=1)
plt.xlabel("date")
plt.ylabel("daily log-return (%)")
plt.title("Daily log-returns hover around zero")
plt.show()

## Step 6 — stylised fact 1: fat tails

Now the first of two famous facts about real markets. If we draw a **histogram** of
the returns (a bar chart of how often each size of move happens) and lay a **normal
distribution** (the textbook bell curve) on top, with the same average and spread,
the real returns have **fatter tails**. Extreme days, crashes and big rallies,
happen far more often than the bell curve predicts.

This is the single most important fact for measuring risk. If you assume a clean
bell curve, you will badly underestimate how often a very bad day happens. (New to
histograms and bell curves? See `primers/what_is_a_distribution.md`.)

We put the counts on a log scale on the vertical axis so the rare tail events are
easy to see.

In [ ]:
# Work in standard-deviation units so the bell curve is easy to draw.
returns_values = log_returns.values
average_return = returns_values.mean()
spread = returns_values.std()
standardised = (returns_values - average_return) / spread

# The histogram of the real, standardised returns.
plt.figure(figsize=(9, 5))
plt.hist(standardised, bins=60, density=True, color="#2E75B6", alpha=0.6,
         label="real returns")

# The normal bell curve, for comparison.
x_values = np.linspace(-6, 6, 300)
bell_curve = (1 / np.sqrt(2 * np.pi)) * np.exp(-x_values ** 2 / 2)
plt.plot(x_values, bell_curve, color="#C0392B", linewidth=3, label="normal bell curve")

# A log scale on the y-axis makes the rare tail events visible.
plt.yscale("log")
plt.xlabel("return (in standard deviations)")
plt.ylabel("how often (log scale)")
plt.title("Fat tails: extreme days happen more than a bell curve predicts")
plt.legend()
plt.show()

## Step 7 — stylised fact 2: volatility clustering

**Volatility** means how much the price swings: the *size* of the moves, ignoring
their direction. In real markets, big moves tend to follow big moves and calm
follows calm. The turbulence comes in **clusters**, not spread out evenly.

A simple way to see it: plot a **rolling** (moving) standard deviation of the
returns. For each day we take the spread of the returns over the previous month or
so. It rises during stormy periods and falls during calm ones, instead of staying
flat.

In [ ]:
# A 21-day rolling standard deviation: for each day, the spread of the returns
# over the previous ~month (21 trading days). High = stormy, low = calm.
rolling_volatility = log_returns.rolling(window=21).std().dropna()

plt.figure(figsize=(9, 4.5))
plt.plot(rolling_volatility.index, rolling_volatility.values * 100, color="#7030A0")
plt.xlabel("date")
plt.ylabel("21-day volatility (%)")
plt.title("Volatility clustering: calm and stormy periods bunch together")
plt.show()

### Stretch (optional) — log-returns really do add up

Skip if you're new. If you'd like to *see* the additive property from step 3, here
is a check. The log-return over the whole period should equal the plain sum of all
the daily log-returns, and it should also match `ln(last price / first price)`. We
confirm all three agree.

In [ ]:
# Sum of every daily log-return.
sum_of_daily = float(log_returns.sum())

# The direct log-return from the first day to the last day.
first_price = float(close_price.iloc[0])
last_price = float(close_price.iloc[-1])
whole_period = float(np.log(last_price / first_price))

print("sum of daily log-returns :", round(sum_of_daily, 6))
print("ln(last / first)         :", round(whole_period, 6))
print("They match - log-returns add up over time, exactly as promised.")

## What you just did

You converted a wandering price into returns, learned why log-returns are the
quant's choice (they add over time and are symmetric), and saw two stylised facts
with your own eyes: fat tails and volatility clustering.

Next notebook: `03_stationarity_acf_pacf.ipynb`, where we use a proper statistical
test to check whether a series is steady enough to model at all.